In [0]:
spark

In [0]:
# startsssssss
from pyspark.sql.functions import *
dbutils.widgets.text('sass','')
sas_token =dbutils.widgets.get('sass')
spark.conf.set(f"fs.azure.account.key.costlowstorage.dfs.core.windows.net", sas_token)


In [0]:
from pyspark.sql.functions import current_timestamp
from pyspark.sql import Row

spark.sql("""
          create table if not exists ingestion_log(
                table_name string,
                run_id string,
                load_type string,
                status string,
                raw_count long,
                processed_rows long,
                raw_path string,
                curated_path string,
                audit_path string,
                start_time timestamp,
                end_time timestamp,
                message string
          )
          using delta
          """)





In [0]:
from pyspark.sql.functions import *
from pyspark.sql.functions import current_timestamp
from pyspark.sql import Row
import json
from pyspark.sql.functions import max as spark_max
from pyspark.sql.types import *
from delta.tables import DeltaTable


dbutils.widgets.text("table_name", "")
dbutils.widgets.text("load_type", "")
dbutils.widgets.text("run_id", "")
dbutils.widgets.text("storage_account", "")
dbutils.widgets.text("raw_container", "")
dbutils.widgets.text("curated_container", "")
dbutils.widgets.text("watermark_value","")
dbutils.widgets.text("file_name","")

storage_account = dbutils.widgets.get("storage_account")
raw_container = dbutils.widgets.get("raw_container")
curated_container = dbutils.widgets.get("curated_container")
table_name = dbutils.widgets.get("table_name")
load_type = dbutils.widgets.get("load_type")
run_id = dbutils.widgets.get("run_id")
watermark_value = dbutils.widgets.get("watermark_value")
file_name = dbutils.widgets.get("file_name")


from datetime import datetime

from pyspark.sql.types import *

log_schema = StructType([
    StructField("table_name", StringType(), True),
    StructField("run_id", StringType(), True),
    StructField("load_type", StringType(), True),
    StructField("status", StringType(), True),
    StructField("raw_count", LongType(), True),
    StructField("processed_rows", LongType(), True),
    StructField("raw_path", StringType(), True),
    StructField("curated_path", StringType(), True),
    StructField("audit_path", StringType(), True),
    StructField("start_time", TimestampType(), True),
    StructField("end_time", TimestampType(), True),
    StructField("message", StringType(), True)
])


def log_sql(status=None, message=None, raw_count=None,
            processed_rows=None, start_time=None, end_time=None,
            table_name=None, run_id=None, load_type=None,
            curated_path=None, audit_path=None, raw_path=None):

    log_data = [(
        table_name,
        run_id,
        load_type,
        status,
        int(raw_count) if raw_count is not None else None,
        int(processed_rows) if processed_rows is not None else None,
        raw_path,
        curated_path,
        audit_path,
        start_time,
        end_time,
        message
    )]

    log_df = spark.createDataFrame(log_data, log_schema)

    log_df.write.format("delta") \
        .mode("append") \
        .saveAsTable("ingestion_log")

                  

start_time = datetime.now()
status = "STARTED"
processed_rows = 0
error_message = None

# logger.info(f"Starting load for table: {table_name}")
# logger.info(f"Run ID: {run_id}")


raw_path = f"abfss://{raw_container}@{storage_account}.dfs.core.windows.net/{table_name}/{file_name}"

curated_path = f"abfss://{curated_container}@{storage_account}.dfs.core.windows.net/{table_name}/fact_{table_name}/"
audit_path = f"abfss://{curated_container}@{storage_account}.dfs.core.windows.net/{table_name}/pipeline_audit/"


from pyspark.sql.functions import *
from delta.tables import DeltaTable
try:
    df_raw = spark.read.format("parquet").load(raw_path)
    raw_count = df_raw.count()
    # logger.info(f"Raw count: {raw_count}")
    log_sql(status='STARTED',start_time=start_time,table_name=table_name,run_id=run_id,load_type=load_type,raw_count=raw_count,processed_rows=processed_rows,curated_path=curated_path,message=f"table load started",
            raw_path=raw_path,audit_path=audit_path)

    df_transformed = (
        df_raw
        .dropDuplicates(["SalesOrderID"])
        .withColumn("OrderDate", to_date("OrderDate"))
        .withColumn("DueDate", to_date("DueDate"))
        .withColumn("ShipDate", to_date("ShipDate"))
        .withColumn("TotalDue", col("TotalDue").cast("double"))
        .withColumn("order_year", year("OrderDate"))
        .withColumn("order_month", month("OrderDate"))
        .withColumn("order_status_desc",
            when(col("Status") == 1, "In Process")
            .when(col("Status") == 2, "Approved")
            .when(col("Status") == 3, "Backordered")
            .when(col("Status") == 4, "Rejected")
            .when(col("Status") == 5, "Shipped")
            .when(col("Status") == 6, "Cancelled")
            .otherwise("Unknown")
        )
        .withColumn("processing_run_id", lit(run_id))
        .withColumn("processing_timestamp", current_timestamp())
    )
    if load_type == "Incremental":
        if watermark_value in [None, "", "null", "None"]:
            watermark_value = None

        if watermark_value is None and DeltaTable.isDeltaTable(spark, curated_path):

            existing_df = spark.read.format("delta").load(curated_path)

            watermark_value = (
                existing_df
                .agg(spark_max("ModifiedDate").alias("max_date"))
                .collect()[0]["max_date"]
            )

            log_sql(status='In progress',start_time=datetime.now(),table_name=table_name,run_id=run_id,load_type=load_type,processed_rows=processed_rows,curated_path=curated_path,message=f"Watermark derived from curated: {watermark_value}",
            raw_path=raw_path)


        if watermark_value is not None:
            df_transformed = df_transformed.filter(
                col("ModifiedDate") > lit(watermark_value)
            )
        else:
            log_sql(status='In progress',start_time=datetime.now(),table_name=table_name,run_id=run_id,load_type=load_type,processed_rows=processed_rows,curated_path=curated_path,message=f"Running full load")


    processed_rows = df_transformed.count()
    log_sql(status='In progress',start_time=datetime.now(),table_name=table_name,run_id=run_id,load_type=load_type,processed_rows=processed_rows,curated_path=curated_path,message=f"Rows after transform {processed_rows}",raw_count=raw_count,raw_path=raw_path)

    if DeltaTable.isDeltaTable(spark, curated_path):
        delta_table = DeltaTable.forPath(spark, curated_path)
        (
            delta_table.alias("target")
            .merge(
                df_transformed.alias("source"),
                "target.SalesOrderID = source.SalesOrderID"
            )
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )
        processed_rows = df_transformed.count()
    else:
        (
            df_transformed.write
            .format("delta")
            .mode("overwrite")
            .partitionBy("order_year", "order_month")
            .option("mergeSchema", "true")
            .save(curated_path)
        )

        processed_rows = df_transformed.count()

    #spark.sql(f"OPTIMIZE delta.`{curated_path}`")
    status = "SUCCESS"
    
    log_sql(status=status,start_time=datetime.now(),table_name=table_name,run_id=run_id,load_type=load_type,processed_rows=processed_rows,message=f"load complted with rows:{processed_rows}",raw_count=raw_count,end_time=datetime.now(),raw_path=raw_path,curated_path=curated_path)
   

except Exception as e:
    status = "FAILED"
    error_message = str(e)
    # logger.error(f"Load failed: {error_message}")
    log_sql(status=status,start_time=datetime.now(),table_name=table_name,run_id=run_id,load_type=load_type,processed_rows=0,message=f"load Failed with error:{error_message}",raw_count=0,end_time=datetime.now(),raw_path=raw_path,curated_path=curated_path)

finally:
    try:

        end_time = datetime.now()

        audit_schema = StructType([
            StructField("table_name", StringType(), True),
            StructField("run_id", StringType(), True),
            StructField("load_type", StringType(), True),
            StructField("status", StringType(), True),
            StructField("raw_count", LongType(), True),
            StructField("processed_rows", LongType(), True),
            StructField("start_time", TimestampType(), True),
            StructField("end_time", TimestampType(), True),
            StructField("error_message", StringType(), True)
        ])

        audit_data = [(
            table_name,
            run_id,
            load_type,
            status,
            int(raw_count) if 'raw_count' in locals() else 0,
            int(processed_rows),
            start_time,
            end_time,
            error_message
        )]

        spark.createDataFrame(audit_data, audit_schema) \
            .write.format("delta") \
            .mode("append") \
            .save(audit_path)

        log_sql(status='Auditing',start_time=datetime.now(),table_name=table_name,run_id=run_id,load_type=load_type,processed_rows=processed_rows,message=f"Auditing complted",raw_path=raw_path,audit_path=audit_path)

    except Exception as audit_error:
        log_sql(status=status,start_time=datetime.now(),table_name=table_name,run_id=run_id,load_type=load_type,processed_rows=0,message=f"audit Failed with error:{str(audit_error)}",raw_count=0,end_time=datetime.now(),raw_path=raw_path,curated_path=curated_path)
        # logger.error(f"Audit logging failed: {str(audit_error)}")

max_modified_date=None
if processed_rows > 0:
    max_modified_date = (
        df_transformed
        .agg(spark_max("ModifiedDate").alias("max_date"))
        .collect()[0]["max_date"]
    )

if max_modified_date is not None:
    max_modified_date = str(max_modified_date)

result = {
    "processed_rows": int(processed_rows),
    "status": status,
    "table_name": table_name,
    "run_id": run_id,
    "max_modified_date": max_modified_date
}

dbutils.notebook.exit(json.dumps(result))

In [0]:
spark.sql(f"""
          select * from ingestion_log order by 1 desc
          """).display()